In [ ]:
!pip -q install -U transformers accelerate bitsandbytes


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 18.6 MB/s eta 0:00:00


In [ ]:
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto").eval()

SYSTEM = (
    "Bạn là biên tập viên tiếng Việt. Nhiệm vụ: sửa lỗi chính tả/dấu câu, "
    "làm câu văn rõ ràng, giữ nguyên ý, không bịa thêm. "
    "Giữ cấu trúc tiêu đề/mục nếu có, có thể xuống dòng và gạch đầu dòng cho dễ đọc. "
    "Chỉ trả về văn bản đã chỉnh sửa."
)

def split_text(s, max_chars=3000):
    s = s.strip()
    chunks = []
    i = 0
    while i < len(s):
        chunks.append(s[i:i+max_chars])
        i += max_chars
    return chunks

@torch.inference_mode()
def polish(chunk):
    messages = [
        {"role":"system","content":SYSTEM},
        {"role":"user","content":chunk}
    ]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=1200, do_sample=False)
    gen = out[0, inputs["input_ids"].shape[1]:]
    return tok.decode(gen, skip_special_tokens=True).strip()

for txt_file in Path(".").glob("*.txt"):
    raw = txt_file.read_text(encoding="utf-8", errors="ignore")
    chunks = split_text(raw, max_chars=3000)
    fixed = []
    for c in chunks:
        fixed.append(polish(c))
    out_path = txt_file.with_name(txt_file.stem + ".fixed.txt")
    out_path.write_text("\n\n".join(fixed).strip(), encoding="utf-8")
    print("Saved:", out_path)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Saved: merged_ALLfile.fixed.txt


In [1]:
input_file = "/content/merged_ALLfile.fixed.txt"
output_file = "finaldata.txt"

with open(input_file, "r", encoding="utf-8") as f:
    text = f.read()

text = text.replace("#", "").replace("*", "")

with open(output_file, "w", encoding="utf-8") as f:
    f.write(text)

print("done")


done
